In [ ]:
import glob
import os
import re
import math
import random
from collections import Counter


class WikipediaCorpus:

  TOKEN_PATTERN = re.compile("[A-Za-zàèéìòùÀÈÉÌÒÙ]+")
  NOISE_TOKENS = {"nbsp", "ndash", "mdash"}

  def __init__(self, language, zip_path, extract_dir, category_prefix, meta_prefixes):
    self.language = language
    self.zip_path = zip_path
    self.extract_dir = extract_dir
    self.category_prefix = category_prefix
    self.meta_prefixes = meta_prefixes

    self.corpus_root = None
    self.files = []

    self.articles = []
    self.redirects = []
    self.categories = []
    self.meta = []

    self.preprocessed = []

    self.word_counter = None
    self.bigram_counter = None
    self.top500_words = []
    self.bottom500_words = []
    self.top500_bigrams = []
    self.bottom500_bigrams = []

    self.lengths = []
    self.articles_stats = []
    self.richness = []

    self.topic_vocabulary = set()
    self.prototypes = {}
    self.labels = {}
    self.assignments = []

  def load(self):
    ! unzip -q {self.zip_path} -d {self.extract_dir}
    self.corpus_root = self._find_corpus_root()
    self.files = self._get_corpus_paths()
    print(f"{self.language.upper()} files: {len(self.files)}")

  def _find_corpus_root(self):
    candidates = glob.glob(os.path.join(self.extract_dir, "*"))
    subfolders = [c for c in candidates if os.path.isdir(c)]
    return subfolders[0]

  def _get_corpus_paths(self):
    dataset = []
    for filename in glob.glob(os.path.join(self.corpus_root, "*.txt")):
      dataset.append(filename)
    return dataset

  def classify(self):
    for filepath in self.files:
      text_type, title, body = self._classify_article(filepath)
      record = (filepath, title, body)
      if text_type == "category":
        self.categories.append(record)
      elif text_type == "meta":
        self.meta.append(record)
      elif text_type == "redirect":
        self.redirects.append(record)
      else:
        self.articles.append(record)

    print(f"=== {self.language.upper()} ===")
    print("Articles:", len(self.articles))
    print("Redirects:", len(self.redirects))
    print("Categories:", len(self.categories))
    print("Meta (non-content namespaces):", len(self.meta))

  def _classify_article(self, filepath):
    with open(filepath, "r") as f:
      lines = f.read().split("\n")

    title = lines[0]
    body = "\n".join(lines[1:]).strip()

    if title.startswith(self.category_prefix):
      return "category", title, body
    if any(title.startswith(prefix) for prefix in self.meta_prefixes):
      return "meta", title, body
    if body.startswith("REDIRECT"):
      return "redirect", title, body
    return "article", title, body

  def preprocess(self):
    self.preprocessed = []
    for filepath, title, body in self.articles:
      cleaned_text = self._strip_category_lines(body)
      cleaned_text = self._strip_section_headers(cleaned_text)
      tokens = self._tokenize(cleaned_text)
      self.preprocessed.append({
          "filepath": filepath,
          "title": title,
          "cleaned_text": cleaned_text,
          "tokens": tokens,
      })
    print(f"{self.language.upper()} articles preprocessed: {len(self.preprocessed)}")

  def _strip_category_lines(self, body):
    lines = body.split("\n")
    cleaned = [line for line in lines if not line.strip().startswith(self.category_prefix)]
    return "\n".join(cleaned)

  def _strip_section_headers(self, body):
    lines = body.split("\n")
    cleaned = [line for line in lines
               if not (line.strip().startswith("==") and line.strip().endswith("=="))]
    return "\n".join(cleaned)

  def _excessive_repetitions(self, token, max_consecutive=2):
    repetitions = 1
    for i in range(1, len(token)):
      if token[i] == token[i - 1]:
        repetitions += 1
        if repetitions > max_consecutive:
          return True
      else:
        repetitions = 1
    return False

  def _tokenize(self, text):
    raw_tokens = self.TOKEN_PATTERN.findall(text.lower())
    return [t for t in raw_tokens
            if t not in self.NOISE_TOKENS and not self._excessive_repetitions(t)]

  def _ngrams(self, tokens, n):
    if type(n) != int or n < 0 or n > len(tokens):
      print("n has to be a positive integer with max value = len(tokens).")
      return []
    return [" ".join(tokens[i:i + n]) for i in range(len(tokens) - n + 1)]

  def _random_distinct_indices(self, total_n, how_many):
    indices = set()
    while len(indices) < how_many and len(indices) < total_n:
      index = random.randint(0, total_n - 1)
      indices.add(index)
    return indices

  def _top_bottom(self, counter, n=500):
    top_n = counter.most_common(n)

    sorted_candidates = sorted(counter.items(), key=lambda pair: pair[1])
    min_freq = sorted_candidates[0][1]
    candidates = [pair for pair in sorted_candidates if pair[1] == min_freq]

    if len(candidates) <= n:
      bottom_n = list(candidates)
      others = [pair for pair in sorted_candidates if pair[1] != min_freq]
      bottom_n.extend(others[:n - len(bottom_n)])
    else:
      chosen_indices = self._random_distinct_indices(len(candidates), n)
      bottom_n = [candidates[i] for i in chosen_indices]

    return top_n, bottom_n

  def compute_word_and_bigram_stats(self):
    all_tokens = []
    all_bigrams = []
    n_too_short = 0

    for article in self.preprocessed:
      tokens = article["tokens"]
      all_tokens.extend(tokens)

      if len(tokens) >= 2:
        all_bigrams.extend(self._ngrams(tokens, 2))
      else:
        n_too_short += 1

    self.word_counter = Counter(all_tokens)
    self.bigram_counter = Counter(all_bigrams)
    print(f"{self.language.upper()} articles too short excluded: {n_too_short}")

    self.top500_words, self.bottom500_words = self._top_bottom(self.word_counter, 500)
    self.top500_bigrams, self.bottom500_bigrams = self._top_bottom(self.bigram_counter, 500)

  def _print_ranking(self, pairs, title):
    print(f"\n=== {title} ===")
    for element, frequency in pairs:
      print(element, frequency)

  def print_word_bigram_rankings(self):
    self._print_ranking(self.top500_words, f"{self.language.upper()} - top 500 words")
    self._print_ranking(self.bottom500_words, f"{self.language.upper()} - bottom 500 words")
    self._print_ranking(self.top500_bigrams, f"{self.language.upper()} - top 500 bigrams")
    self._print_ranking(self.bottom500_bigrams, f"{self.language.upper()} - bottom 500 bigrams")

  def _manual_average(self, numbers):
    total = 0
    for x in numbers:
      total += x
    return total / len(numbers)

  def _manual_median(self, numbers):
    ordered = sorted(numbers)
    n = len(ordered)
    mid = n // 2
    if n % 2 == 1:
      return ordered[mid]
    return (ordered[mid - 1] + ordered[mid]) / 2

  def compute_length_stats(self):
    self.lengths = [len(article["tokens"]) for article in self.preprocessed]

    self.articles_stats = []
    for article in self.preprocessed:
      tokens = article["tokens"]
      n_tokens = len(tokens)
      n_distinct = len(set(tokens))
      richness = n_distinct / n_tokens if n_tokens > 0 else 0
      self.articles_stats.append({
          "title": article["title"],
          "n_tokens": n_tokens,
          "n_distinct": n_distinct,
          "richness": richness,
      })
    self.richness = [r["richness"] for r in self.articles_stats]

  def print_length_summary(self, bin_width=100, max_bar_width=60):
    print(f"--- Article length stats ({self.language.upper()}) ---")
    print("Number of articles:", len(self.lengths))
    print("Minimum:", min(self.lengths))
    print("Maximum:", max(self.lengths))
    print("Average:", round(self._manual_average(self.lengths), 2))
    print("Median:", self._manual_median(self.lengths))

    self._print_text_histogram(bin_width, max_bar_width)

    print("Average richness:", round(self._manual_average(self.richness), 3))

  def _print_text_histogram(self, bin_width, max_bar_width):
    bin_counts = {}
    for length in self.lengths:
      bin_index = length // bin_width
      if bin_index not in bin_counts:
        bin_counts[bin_index] = 0
      bin_counts[bin_index] += 1

    max_count = max(bin_counts.values())

    print(f"\n--- Article length histogram ({self.language.upper()}), {bin_width} tokens ---")
    for bin_index in sorted(bin_counts.keys()):
      count = bin_counts[bin_index]
      bar_length = math.ceil((count / max_count) * max_bar_width)
      bottom = bin_index * bin_width
      top = bottom + bin_width - 1
      print(f"{bottom:6d}-{top:<6} | {'#' * bar_length} ({count})")

  def _compute_document_frequency(self, preprocessed):
    df = Counter()
    for article in preprocessed:
      df.update(set(article["tokens"]))
    return df

  def _build_topic_vocabulary(self, document_frequency, n_total_articles, min_ratio=0.002, max_ratio=0.05):
    minimum = n_total_articles * min_ratio
    maximum = n_total_articles * max_ratio
    return {word for word, count in document_frequency.items() if minimum <= count <= maximum}

  def _article_signature(self, tokens, size=15):
    local = Counter(tokens)
    candidate = [word for word, _ in local.most_common(200) if word in self.topic_vocabulary]
    return set(candidate[:size])

  def _set_similarity(self, set_a, set_b):
    if not set_a or not set_b:
      return 0.0
    intersection = len(set_a & set_b)
    union = len(set_a | set_b)
    return intersection / union

  def discover_and_assign_domains(self, sample_size=3000, threshold=0.10,
                                   signature_size=15, profile_size=25, top_k_prototypes=70):
    document_frequency = self._compute_document_frequency(self.preprocessed)
    self.topic_vocabulary = self._build_topic_vocabulary(document_frequency, len(self.preprocessed))
    print(f"Topic vocabulary size {self.language.upper()}: {len(self.topic_vocabulary)}")

    self.prototypes, self.labels = self._bootstrap_prototypes(
        sample_size, threshold, signature_size, profile_size, top_k_prototypes
    )
    print(f"Prototypes found {self.language.upper()}: {len(self.prototypes)}")

    self.assignments = self._assign_to_prototypes(threshold, signature_size)

  def _bootstrap_prototypes(self, sample_size, threshold, signature_size, profile_size, top_k_prototypes):
    sample_indices = self._random_distinct_indices(len(self.preprocessed), sample_size)
    sample = [self.preprocessed[i] for i in sample_indices]

    profiles = {}
    profile_sets = {}
    sizes = {}

    for article in sample:
      signature = self._article_signature(article["tokens"], signature_size)

      best_cluster = None
      best_similarity = 0.0
      for cluster_index, profile_set in profile_sets.items():
        similarity = self._set_similarity(signature, profile_set)
        if similarity > best_similarity:
          best_similarity = similarity
          best_cluster = cluster_index

      if best_cluster is not None and best_similarity >= threshold:
        profiles[best_cluster].update(signature)
        profile_sets[best_cluster] = {w for w, _ in profiles[best_cluster].most_common(profile_size)}
        sizes[best_cluster] += 1
      else:
        new_index = len(profiles)
        profiles[new_index] = Counter(signature)
        profile_sets[new_index] = set(signature)
        sizes[new_index] = 1

    sorted_clusters = sorted(sizes.items(), key=lambda pair: pair[1], reverse=True)
    top_k_indices = [index for index, _ in sorted_clusters[:top_k_prototypes]]

    prototypes = {index: profile_sets[index] for index in top_k_indices}
    labels = {
        index: [w for w, _ in profiles[index].most_common(8)]
        for index in top_k_indices
    }
    return prototypes, labels

  def _assign_to_prototypes(self, threshold, signature_size):
    assignments = []
    for article in self.preprocessed:
      signature = self._article_signature(article["tokens"], signature_size)

      best_match = "unclassified"
      best_similarity = threshold
      for prototype_index, profile in self.prototypes.items():
        similarity = self._set_similarity(signature, profile)
        if similarity > best_similarity:
          best_similarity = similarity
          best_match = prototype_index

      assignments.append(best_match)
    return assignments

  def print_domain_summary(self, n=3, min_size=20):
    counts = Counter(self.assignments)
    real_domains = Counter({k: v for k, v in counts.items() if k != "unclassified"})
    significant_domains = Counter({k: v for k, v in real_domains.items() if v >= min_size})
    n_discarded = len(real_domains) - len(significant_domains)
    sorted_domains = significant_domains.most_common()

    lang = self.language.upper()
    print(f"\n--- {lang}: unclassified articles: {counts['unclassified']} out of {sum(counts.values())} ---")
    print(f"--- {lang}: clusters discarded as too small (< {min_size} articles): {n_discarded} ---")
    print(f"--- {lang}: top {n} domains ---")
    for index, count in sorted_domains[:n]:
      print(f"Domain (cluster {index}) - {self.labels[index]}: {count} articles")
    print(f"--- {lang}: bottom {n} domains ---")
    for index, count in sorted_domains[-n:]:
      print(f"Domain (cluster {index}) - {self.labels[index]}: {count} articles")


def compare_corpora(corpora, min_size=20):
  languages = list(corpora.keys())

  def print_header():
    row = f"{'':25s}"
    for lang in languages:
      row += f"{lang.upper():>12s}"
    print(row)

  def print_row(label, values):
    row = f"{label:25s}"
    for value in values:
      row += f"{value:>12}"
    print(row)

  print("--- Corpus composition ---")
  print_header()
  print_row("Regular articles", [len(corpora[lang].articles) for lang in languages])
  print_row("Redirects", [len(corpora[lang].redirects) for lang in languages])
  print_row("Category pages", [len(corpora[lang].categories) for lang in languages])
  print_row("Meta pages", [len(corpora[lang].meta) for lang in languages])

  print("\n--- Vocabulary size ---")
  print_header()
  print_row("Distinct words", [len(corpora[lang].word_counter) for lang in languages])
  print_row("Distinct bigrams", [len(corpora[lang].bigram_counter) for lang in languages])

  print("\n--- Article length ---")
  print_header()
  print_row("Average", [round(corpora[lang]._manual_average(corpora[lang].lengths), 1) for lang in languages])
  print_row("Median", [corpora[lang]._manual_median(corpora[lang].lengths) for lang in languages])
  print_row("Minimum", [min(corpora[lang].lengths) for lang in languages])
  print_row("Maximum", [max(corpora[lang].lengths) for lang in languages])

  print("\n--- Lexical richness (distinct/total tokens) ---")
  print_header()
  print_row("Average richness",
             [round(corpora[lang]._manual_average(corpora[lang].richness), 3) for lang in languages])

  print("\n--- Domain clustering coverage ---")
  print_header()
  unclassified_ratios = []
  significant_counts = []
  top3_coverages = []
  for lang in languages:
    corpus = corpora[lang]
    counts = Counter(corpus.assignments)
    total = sum(counts.values())
    significant = Counter({k: v for k, v in counts.items() if k != "unclassified" and v >= min_size})
    unclassified_ratios.append(round(counts["unclassified"] / total, 3))
    significant_counts.append(len(significant))
    top3_coverages.append(round(sum(c for _, c in significant.most_common(3)) / total, 3))
  print_row("Unclassified ratio", unclassified_ratios)
  print_row("Significant clusters", significant_counts)
  print_row("Top-3 coverage", top3_coverages)

In [ ]:
# --- How to add a new language ---
#
# 1. Add a new entry to the dictionary below, one key per language code
#    (e.g. "es" for Spanish). No other code needs to change.
#
# 2. Each entry needs four things:
#    - "zip_path": the corpus archive uploaded to this Colab session
#      (e.g. "wiki.es.50k.zip")
#    - "extract_dir": any folder name to extract it into (must be
#      different from the other languages' folders)
#    - "category_prefix": how the new language's Wikipedia marks a
#      category page in the title (e.g. "Categoría:" for Spanish) --
#      check one category-page file in the new corpus to confirm it
#    - "meta_prefixes": the list of non-content namespace prefixes in
#      that language (Wikipedia:, Template:, File:, Talk:, User:, ...
#      translated). These are documented, fixed lists, the same for
#      every Wikipedia edition:
#      https://en.wikipedia.org/wiki/Wikipedia:Namespace
#      (open the equivalent page in the new language's Wikipedia to
#      get the translated prefixes)
#
# Example for Spanish:
# "es": {
#     "zip_path": "wiki.es.50k.zip",
#     "extract_dir": "corpus_es",
#     "category_prefix": "Categoría:",
#     "meta_prefixes": [
#         "Wikipedia:", "Plantilla:", "Archivo:", "Portal:",
#         "Discusión:", "Usuario:", "Ayuda:", "Módulo:", "MediaWiki:",
#     ],
# },
#
# Everything downstream (classification, preprocessing, word/bigram
# stats, length stats, domain clustering, and the final comparison
# table) works automatically for any number of languages, because it
# all iterates over the "corpora" dictionary rather than referring to
# "en"/"it" by name.

languages_config = {
    "en": {
        "zip_path": "wiki.en.50k.zip",
        "extract_dir": "corpus_en",
        "category_prefix": "Category:",
        "meta_prefixes": [
            "Wikipedia:", "Template:", "File:", "Portal:",
            "Talk:", "User:", "Help:", "Module:", "Draft:", "MediaWiki:",
        ],
    },
    "it": {
        "zip_path": "wiki.it.50k.zip",
        "extract_dir": "corpus_it",
        "category_prefix": "Categoria:",
        "meta_prefixes": [
            "Wikipedia:", "Template:", "File:", "Portale:", "Progetto:",
            "Discussione:", "Utente:", "Aiuto:", "Modulo:", "MediaWiki:",
        ],
    },
}

corpora = {}
for lang, config in languages_config.items():
  corpus = WikipediaCorpus(
      language=lang,
      zip_path=config["zip_path"],
      extract_dir=config["extract_dir"],
      category_prefix=config["category_prefix"],
      meta_prefixes=config["meta_prefixes"],
  )
  corpus.load()
  corpus.classify()
  corpus.preprocess()
  corpus.compute_word_and_bigram_stats()
  corpus.compute_length_stats()
  corpus.discover_and_assign_domains()
  corpora[lang] = corpus

for lang, corpus in corpora.items():
  corpus.print_word_bigram_rankings()
  corpus.print_length_summary()
  corpus.print_domain_summary()

compare_corpora(corpora)

EN files: 54281
=== EN ===
Articles: 20209
Redirects: 22919
Categories: 3861
Meta (non-content namespaces): 7292
EN articles preprocessed: 20209
EN articles too short excluded: 121
Topic vocabulary size EN: 11272
Prototypes found EN: 70
IT files: 50000
=== IT ===
Articles: 29096
Redirects: 8166
Categories: 5931
Meta (non-content namespaces): 6807
IT articles preprocessed: 29096
IT articles too short excluded: 529
Topic vocabulary size IT: 9838
Prototypes found IT: 70

=== EN - top 500 words ===
the 718499
of 363035
and 292987
in 290227
a 220504
to 201723
was 110312
s 91325
is 90099
for 86518
on 84940
as 84326
by 75061
with 73346
at 56093
he 55625
that 55493
from 53954
his 46304
it 44220
an 39112
were 31148
are 29763
which 28975
or 26489
this 25723
be 24900
first 24097
also 24074
new 23867
had 22371
has 21095
one 20993
their 19556
after 19381
its 18845
not 18710
who 18075
but 17723
two 17056
they 16781
have 15980
her 15589
th 15567
all 14538
other 14370
she 14271
been 13996
time 13727
m